# Batch Normalisation

From our findings above, at initialisation we want our linear activations (x @ w + b) to be roughly standard normal since this will ensure stable nonlinear activations (relu, tanh etc). Batch normalisation proposes that instead of worrying about our setup, why don't we just sample from the standard normal curve anyway by normalising the data (subtract mean, divide std dev)

**Important Note** we only want standard normal at initialisation, not necessarily throughout the training process. We want the model to figure out the distribution as it trains. We implement this by adding a gain and bias to our normalisation formula in the form of gain * ((x*w-mean)/stddev) + bias. Gain is initialised as 1, bias initialised as 0, and these then become some more parameters to backpropagate and optimise. This setup means that at initialisation, the raw batches are normalised to standard normal and multiplied by 1 and added to 0. Then as the training goes on, the raw batches are still standardised which means they are stable, but they are allowed to move around using the gain and bias terms through training

**Another Important Note** Batchnorm has some strange implications for training. In a network without it, a single training example gets fed through, and we only have to compute the loss and gradient as a function of that example. When you introduce batchnorm, you scale it by the mean and std dev of the entire batch, which is a function of many randomly selected observations in the batch. So for any given training example, we may end up with one or more different activations depending on the batch it gets passed in with. This can add a slight bit of jitteriness or instability to the training. While this may sound like a bad thing, this actually makes it harder for the network to overfit to any given training examples, since every time they go through the network, they change slightly due to the different batch mean and std dev.

In summary, batch norm introduces essentially another layer, after some sort of multiplication or linear layer. The purpose is to standardise our activations into some kind of standard normal for the sake of stability. It introduces 4 parameters:
- **batch norm mean**: for every batch passed into training, we capture the mean in order to scale the batch activations to standard normal. Initialise at 0, and keep a running score so we have a sort of full training mean at the end to be used for inference. Gradient not required, since we do not optimise or use gradient descent, we do the training ourselves
- **batch norm std dev**: very similar to above, except we initialise at 1. Optional: we add some epsilon to ensure we never divide by 0
- **batch norm gain**: After we scale the activation to standard normal, we use the following: gain * ((x*w-mean)/stddev) + bias. Initalising at 1 means at the start of training our activations are perfectly standard normal, but we optimise this using gradient descent to let the activation distribution adapt to the batches. These make up part of the parameters to train along with our normal weights, biases and embeddings
- **batch norm bias**: Similar to above, except we initialise at 0 to maintain standard normal at initialisation. These make up part of the parameters to train along with our normal weights, biases and embeddings. This can take the place of our normal activation bias

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [2]:
# Start with our previous MLP
words = open('names.txt', 'r').read().splitlines()
chars = sorted(list(set(''.join(words))))
s_to_i = {s: i+1 for i, s in enumerate(chars)}
s_to_i['.'] = 0
i_to_s = {i:s for s, i in s_to_i.items()}

In [3]:
block_size = 3

# create the dataset, now we have 228146 datapoints
# implement test and training splits
def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        for ch in w+'.':
            ix = s_to_i[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

# create the datasets
X_train, Y_train = build_dataset(words[:n1])
X_val, Y_val = build_dataset(words[n1:n2])
X_test, Y_test = build_dataset(words[n2:])

In [ ]:
emb_dim = 10
hidden_layer_size = 200

# initialise
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, emb_dim),                              generator=g)
W1 = torch.randn((emb_dim * block_size, hidden_layer_size), generator=g) * (5/3) / (emb_dim * block_size)**0.5
# b1 = torch.randn(hidden_layer_size,                         generator=g) * 0.01 we no longer need any bias in the hidden layer - when we take the mean and subtract it, this ends up having no impact, plus we have a batchnorm bias anyway
W2 = torch.randn((hidden_layer_size, 27),                   generator=g) * (5/3) / (hidden_layer_size)**0.5
b2 = torch.randn(27,                                        generator=g) * 0

# initialise gain and bias for the batch norm: setting gain to 1 and bias to 0 means initially that the preactivations will be scaled to standard normal, and then as the model trains it can deviate
batch_norm_gain = torch.ones((1, hidden_layer_size))
batch_norm_bias = torch.zeros((1, hidden_layer_size))

# create the empty tensor to store the running batch means and std dev
batch_mean_running = torch.zeros((1, hidden_layer_size))
batch_std_running = torch.ones((1, hidden_layer_size))
# note these wont be optimised, so we don't put them in parameters, these are just used to store our running totals, no impact on model training

params = [C, W1, batch_norm_gain, batch_norm_bias, W2, b2]
for p in params:
    p.requires_grad = True

In [ ]:
iterations = 200000 
batch_size = 32

for k in range(iterations):

    # set up minibatching
    ix = torch.randint(0, X_train.shape[0], (batch_size, ))
    Xb, Yb = X_train[ix], Y_train[ix]

    # forward
    emb = C[Xb]
    emb_cat = emb.view(emb.shape[0], -1)
    hpreact = emb_cat @ W1 + # b1 we no longer need any bias - when we take the mean and subtract it, this ends up having no impact, plus we have a batchnorm bias anyway

    # here we will implement the batch norm standardisation
    # get mean and std dev for each neuron across the batch
    # also we will calculate the full training mean and std dev as we go, rather than a separate step at the end
    batch_mean_i = hpreact.mean(0, keepdim=True)
    batch_std_i = hpreact.std(0, keepdim=True)

    # normalise
    normal_h = (hpreact - batch_mean_i) / batch_std_i
    
    # scale with gain and bias
    scaled_h = batch_norm_gain * normal_h + batch_norm_bias

    # the incremental updates to the overall training mean and std dev as we go along - set no_grad cos got nothing to do with training
    # slightly adjust the running totals in the way of the batch mean and std dev for every iteration
    with torch.no_grad():
        batch_mean_running = 0.999 * batch_mean_running + 0.001 * batch_mean_i # this 0.001 is known as momentum - we dont want to set too high if our batches are small, since our running 'training' might not converge nicely
        batch_std_running = 0.999 * batch_std_running + 0.001 * batch_std_i

    # then we activate
    h = torch.tanh(scaled_h) 
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)
    
    # backward
    for p in params: # flush
        p.grad = None
    loss.backward()

    # gradient descent
    lr = 0.1 if k < 100000 else 0.01
    for p in params:
        p.data += -lr * p.grad

    if k % 10000 == 0:
        print(f'{k:7d}/{iterations:7d}: {loss.item():.4f}')


      0/ 200000: 4.0458
  10000/ 200000: 2.8717
  20000/ 200000: 2.1297
  30000/ 200000: 2.3604
  40000/ 200000: 2.1621
  50000/ 200000: 1.9964
  60000/ 200000: 1.9766
  70000/ 200000: 2.0733
  80000/ 200000: 1.8734
  90000/ 200000: 2.4865
 100000/ 200000: 2.1098
 110000/ 200000: 2.4758
 120000/ 200000: 2.2203
 130000/ 200000: 1.6957
 140000/ 200000: 1.8019
 150000/ 200000: 2.2199
 160000/ 200000: 1.7706
 170000/ 200000: 1.9780
 180000/ 200000: 2.1442
 190000/ 200000: 1.9827


In [6]:
# lets look at the hpreact for the last batch in the training pass

hpreact.shape # this is batch size x hidden_layer_size - so for each x in the batch, it has the 200 raw activations

h_mean = hpreact.mean(0, keepdim=True) # this agg over the 0th dimension, so in this case the xs, giving you the average activation for each neuron across the entire batch
h_std = hpreact.std(0, keepdim=True) # same thing for std dev
# these are now both 1 x hidden_layer_size

# broadcasting rules
# h:   32, 200
# mean: 1, 200

In [ ]:
# however, implementing batch norm means our model now expects a batch of inputs, since we need to calculate some sort of mean and stddev, we can no longer just pass in one input to forward pass
# to get around this, we can calculate the mean and std dev for the entire training set, and apply it when we feed an input forward just to sample
'''with torch.no_grad():
    emb = C[X_train]
    emb_cat = emb.view(emb.shape[0], -1)
    hpreact = emb_cat @ W1 + b1

    # get the mean and std dev for the whole training set
    full_mean = hpreact.mean(0, keepdim=True)
    full_std = hpreact.std(0, keepdim=True)'''
# also when evaluating on the val and test sets, you would need to use the training mean and std dev to avoid data leakage

# Note: we no longer need this step, since we calculate full_mean and full_std incrementally as we train

In [9]:
@torch.no_grad() # disable gradient tracking for the one off evaluation forward pass
def show_split_loss(split):
    x, y = {
        'train': (X_train, Y_train),
        'val': (X_val, Y_val),
        'test': (X_test, Y_test)
    }[split]

    emb = C[x]
    emb_cat = emb.view(emb.shape[0], -1)
    hpreact = emb_cat @ W1 #+ b1

    # implement the same batch norm down here
    normal_h = (hpreact - batch_mean_running) / batch_std_running # use the full training set mean and std dev
    scaled_h = batch_norm_gain * normal_h + batch_norm_bias
    h = torch.tanh(scaled_h)

    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y)
    print(split, loss.item())

show_split_loss('train')
show_split_loss('val')

train 2.071033477783203
val 2.110819101333618
